# 10 · Reranking — the stage that actually reads the candidate

Retrieval that works is **three** stages, not two: retrieve wide and cheap, then
rerank a bounded top-N by *reading* each candidate against the query, then keep
`k`. Notebook 09 is stage one — a cosine, computed over `real[]` columns. This
one is stage two.

A dense score is a distance between two embeddings, which is a summary of a
summary: it knows the candidate is *about databases*, not that it is about the
**query planner**. A reranker reads the text and can tell. That is the whole
argument, and everything below is the shape it takes in a graph:

- `vector_search(rerank=…)` — a flat hit list, reordered, with `rerank_score`
  reported **beside** the unchanged `similarity`;
- `Start(rerank=…)` — better seeds, chosen before any `Hop` walks;
- `Hop(rerank=…)` — a **node plus how it was reached**, which is the part a
  ported flat-list reranker structurally cannot offer;
- `candidates` (the input bound) against `k`/`keep` (the output bound), and why
  pruning early compounds;
- the refusals, each executed, and the jq subset a *model* is allowed to write.

**Everything here runs offline.** A real reranker is a network call, and CI
executes these notebooks against a database with no network — so the reranker
below is a small deterministic stand-in, exactly as notebook 09 uses a two-line
embedding model. It is a term-overlap score, not a cross-encoder, and it is
honest about that: the point is that every ranking on this page can be checked
by reading rather than believed. It doubles as proof that the plain-callable
path is first class — `Rerank(callable)` is not a fallback.

## Setup

The usual seven-node demo graph, with two properties added to the five people: a
short `interests` phrase (which gets **embedded**) and a `bio` sentence (which
gets **read**). That split is the whole notebook in miniature — the embedding
sees the topic, the reranker sees the sentence.

In [1]:
import logging
import textwrap

from demo_graph import arrows, connect, names, seed
from hopai import Hop, Near, Rerank, RerankError, Start, Vector

graph = connect("nb_10_reranking")
ids = seed(graph)

#: interests -> embedded into the `interests` vector field
#: bio       -> read by the reranker
#: tags      -> a list, for the projection section (Erin has none)
PEOPLE = {
    "Alice": ("postgres", "Runs the platform; has not touched a query plan since 2019.",
              ["platform", "oncall"]),
    "Bob": ("design", "Designs the interfaces people ask questions through.", ["design"]),
    "Carol": ("finance", "Writes the statistics reports finance lives on.",
              ["finance", "reporting"]),
    "Dave": ("postgres outdoors", "Left a Postgres cluster running while he climbs.",
             ["oncall"]),
    "Erin": ("postgres design",
             "Reads the planner source for fun; writes up what EXPLAIN and the "
             "statistics mean.", None),
}
for name, (interests, bio, tags) in PEOPLE.items():
    row = {"interests": interests, "bio": bio}
    if tags:
        row["tags"] = tags
    graph.update_nodes(where={"name": name}, set=row)

#: Notebook 09's trick, again: one axis per topic word the "model" knows.
#: It sees the TOPIC and not the sentence -- which is precisely the gap the
#: reranker closes further down.
TOPICS = ["postgres", "design", "finance", "outdoors"]


def topic_model(texts):
    return [[float(topic in text.lower()) for topic in TOPICS] for text in texts]


graph.define_vectors(nodes=[Vector("interests", 4, embed=topic_model)], migrate=True)
print(graph.embed_stale(node_fields=["interests"])["nodes"]["interests"])

# Every provider call is logged to the `hopai.rerankers` logger -- sizes at
# DEBUG, a spent call at WARNING. An application configures logging; a notebook
# silences it so the printed output below is only what the cells printed.
logging.getLogger("hopai.rerankers").addHandler(logging.NullHandler())

{'embedded': ['1', '2', '3', '4', '5'], 'skipped': ['6', '7']}


## The reranker

One method is the whole contract:

```
score(query: str, documents: list[str]) -> list[float]
```

One float per document, **in the order the documents were given**, higher meaning
more relevant. That is all a reranker author implements: which step of a
traversal produced these candidates, whether they came from a cosine or a
lexical match, how they were deduplicated — none of it reaches here, which is
exactly what lets hopai build documents differently at a seed and at a hop
without any reranker knowing.

A Cohere or Voyage client (`model=` required), a `sentence-transformers`
`CrossEncoder`, anything with `.score(query, documents)`, or a plain callable are
all spellings of that one method. Here it is a plain callable, so this page runs
with no network:

In [2]:
def _words(text):
    return {word.strip(".,;:'\"()").lower() for word in text.split()}


def relevance(query, documents):
    """A stand-in for a cross-encoder: how much of the query the document covers.

    A real reranker reads both sides with a model. This counts shared words --
    deterministic, offline, and checkable by eye, which is the only reason it is
    here. It obeys the contract exactly: one float per document, in order.
    """
    asked = _words(query)
    return [len(asked & _words(document)) / len(asked) for document in documents]


QUESTION = "postgres planner statistics explain"

print(relevance(QUESTION, ["a note about the planner and its statistics",
                           "a note about lunch"]))

[0.5, 0.0]


## What the dense stage gets wrong

Ask who can help with a query-planner question. The embedding only sees each
person's `interests`, so Alice — whose interests say *postgres* and nothing else
— is a perfect cosine match:

In [3]:
for hit in graph.vector_search(Near("interests", QUESTION), k=5):
    print(f"{hit['properties']['name']:<6} {hit['similarity']:.3f}")

Alice  1.000
Dave   0.707
Erin   0.707
Bob    0.000
Carol  0.000


A tidy ranking, and the wrong one. Alice's own bio says she has not touched a
query plan since 2019; Erin — second, on a lower cosine — reads the planner
source for fun. The distance cannot know that, because the sentence never
reached it. So hand the same query a stage that reads:

In [4]:
reader = Rerank(relevance,
                document_from='.properties.name + ": " + .properties.bio',
                candidates=5)
print(reader, "\n")

for hit in graph.vector_search(Near("interests", QUESTION), rerank=reader, k=3):
    print(f"{hit['properties']['name']:<6} rerank {hit['rerank_score']:.2f}"
          f"   similarity {hit['similarity']:.3f}")

Rerank(relevance, document_from='.properties.name + ": " + .properties.bio', candidates=5) 

Erin   rerank 0.75   similarity 0.707
Carol  rerank 0.25   similarity 0.000
Dave   rerank 0.25   similarity 0.707


Four things in those three lines are the whole result contract:

- **Erin is first**, on a cosine that ranked her second. That is the reranker
  doing the one job the cosine could not.
- **`similarity` is untouched.** It keeps the value the retrieval stage gave it
  rather than being overwritten, so you can see *what the reranker changed* —
  Carol was promoted from `0.000`, dead last on similarity, because her bio is
  the statistics one.
- **`rerank_score` is additive.** Every other key of a hit is exactly what
  notebook 09 documented; without `rerank=`, results are byte-identical to
  before this feature existed.
- **`candidates` fetched 5 rows, `k` returned 3.** Two different bounds, and
  they never overlap — more on that below.

The provider call happens **after the SQL round trip has closed**, never inside
it: an HTTP call in an open transaction holds a snapshot for a network round
trip, which is the same rule `set_vectors()` keeps for row locks.

## `document_from=` is a rule, not a document

Nothing about the documents exists when the query is written, so the parameter
holds a **jq filter** that hopai evaluates once per candidate at execution time.
jq rather than a projection language invented here, because it is a syntax every
model has already seen ten thousand times — and because its own operators
already cover nested fields, lists and defaults.

The filter runs against the candidate's JSON: the same dict `vector_search()`
returns, plus `paths` at a traversal hop. `build_documents()` is that step on its
own, which is how you check a filter without spending a provider call:

In [5]:
candidates = graph.vector_search(Near("interests", QUESTION), k=5)
print(sorted(candidates[0]), "<- one candidate's keys\n")

for document in reader.build_documents(candidates):
    print(" *", document)

['boosts', 'id', 'properties', 'similarities', 'similarity'] <- one candidate's keys

 * Alice: Runs the platform; has not touched a query plan since 2019.
 * Dave: Left a Postgres cluster running while he climbs.
 * Erin: Reads the planner source for fun; writes up what EXPLAIN and the statistics mean.
 * Bob: Designs the interfaces people ask questions through.
 * Carol: Writes the statistics reports finance lives on.


The projection is jq's, not ours — nested paths, an iterated list, and the
alternative operator for a default all come for free. Erin has no `tags` key at
all, which is what `// ["untagged"]` is answering:

In [6]:
tagged = Rerank(relevance,
                document_from='.properties.name + " [" + '
                              '((.properties.tags // ["untagged"]) | join(", ")) + "]"',
                candidates=5)
for document in tagged.build_documents(candidates):
    print(" *", document)

 * Alice [platform, oncall]
 * Dave [oncall]
 * Erin [untagged]
 * Bob [design]
 * Carol [finance, reporting]


Without the default, that same filter has nothing to say about Erin — and it
**refuses**, naming the filter and the candidate, rather than scoring her against
an empty document:

In [7]:
try:
    Rerank(relevance, document_from='.properties.tags | join(", ")',
           candidates=5).build_documents(candidates)
except ValueError as exc:
    print(f"ValueError: {exc}")

ValueError: document_from='.properties.tags | join(", ")' failed on candidate id='5' -- ValueError: Cannot iterate over null (null). Refusing rather than scoring that candidate against an empty document


Inline the filter at the call site. `document_from=doc` hides the only part a
reader of the query needs to see.

The jq binding is an optional extra — `pip install "hopai[rerankers]"`. The
dependency is `jq` and not a provider: reranker clients are duck-typed exactly
like embedding clients, so nothing here imports one.

## `candidates` bounds the input; `k`/`keep` bounds the output

`candidates` decides how many rows are fetched and how many documents the
provider is billed for. `k` decides how many come back. Reranking cannot promote
a candidate it never saw, so **pruning early is unrecoverable** — with
`candidates=2` the answer is simply out of reach:

In [8]:
for size in (2, 5):
    narrow = Rerank(relevance, document_from='.properties.name + ": " + .properties.bio',
                    candidates=size)
    top = graph.vector_search(Near("interests", QUESTION), rerank=narrow, k=1)[0]
    print(f"candidates={size}: {top['properties']['name']:<6} {top['rerank_score']:.2f}")

candidates=2: Dave   0.25
candidates=5: Erin   0.75


`candidates=2` fetched Alice and Dave, the two best cosines, and the best of a
bad pair is still a bad answer. That is the argument for a **generous
`candidates`**, and it compounds in a traversal: prune wrong at hop 1 and hop 2
never sees the right nodes.

The two numbers are never reconciled silently. On a flat search the order *is*
the answer, so `candidates == k` still reorders and only a smaller `candidates`
refuses. Inside a traversal `candidates` has to be strictly greater than `keep`,
because a subgraph discards the order: rerank exactly as many as survive and the
same nodes continue the walk whichever score sorted them, every document billed
for a guaranteed no-op. Both are refusals rather than clamps — see below —
because clamping would hide that the query's own numbers disagree.

## Step-wise: reranking inside the walk

`Start(rerank=)` re-scores the **seed** set before any `Hop` runs: `near` picks
`candidates` seeds cheaply, the reranker reorders them, `keep` truncates, and
everything downstream is unaware it happened. `keep` is not optional here —
a traversal returns a subgraph, so truncation is the only thing a reranker can
change, and a step that reranks without one refuses. The walk simply starts
somewhere better:

In [9]:
friends = Hop(via={"kind": "friend"})

dense = graph.traverse(Start(near=Near("interests", QUESTION), keep=1), friends)
read = graph.traverse(
    Start(near=Near("interests", QUESTION), keep=1,
          rerank=Rerank(relevance, document_from='.properties.bio', candidates=5)),
    friends)

print("dense seed:", names(dense), arrows(dense))
print("read seed: ", names(read), arrows(read))

dense seed: ['Alice', 'Bob', 'Carol'] ['Alice -friend-> Bob', 'Alice -friend-> Carol']
read seed:  ['Alice', 'Erin'] ['Erin -friend-> Alice']


Same query, same `keep=1`, a different graph comes back: the cosine seeds at
Alice and walks to her friends, the reranker seeds at Erin and walks to hers.

At a **hop** the candidate is not a row — it is *a node plus how it was reached*.
"This person, reached because the person who writes the statistics reports knows
them" is strictly more than "this person", and a reranker can use it. So at a hop
the candidate JSON carries `paths`, and `document_from` may read it.

The reranker below prints every document it is given, which is the clearest way
to see what a hop candidate actually is:

In [10]:
def show(query, documents):
    """relevance(), printing what the reranker was handed."""
    for document in documents:
        print(textwrap.fill(document, 78, initial_indent=" * ", subsequent_indent="   "))
    return relevance(query, documents)


ROUTED = ('.properties.name + ": " + .properties.bio + " -- reached from "'
          ' + (.paths | map(.[-1].properties.name + " (" + .[-1].properties.bio + ")")'
          ' | join(", "))')

own = Hop(via={"kind": "friend"}, hops=(1, 2), near=Near("interests", QUESTION), keep=1,
          rerank=Rerank(show, document_from='.properties.name + ": " + .properties.bio',
                        candidates=5))
routed = Hop(via={"kind": "friend"}, hops=(1, 2), near=Near("interests", QUESTION), keep=1,
             rerank=Rerank(show, document_from=ROUTED, candidates=5))

print("--- the node's own properties")
by_own = graph.traverse(Start(where={"name": "Alice"}), own)
print("kept:", names(by_own), arrows(by_own), "\n")

print("--- the node, plus how it was reached")
by_route = graph.traverse(Start(where={"name": "Alice"}), routed)
print("kept:", names(by_route), arrows(by_route))

--- the node's own properties
 * Bob: Designs the interfaces people ask questions through.
 * Carol: Writes the statistics reports finance lives on.
 * Dave: Left a Postgres cluster running while he climbs.
kept: ['Alice', 'Carol'] ['Alice -friend-> Carol'] 

--- the node, plus how it was reached


 * Bob: Designs the interfaces people ask questions through. -- reached from
   Alice (Runs the platform; has not touched a query plan since 2019.)
 * Carol: Writes the statistics reports finance lives on. -- reached from
   Alice (Runs the platform; has not touched a query plan since 2019.)
 * Dave: Left a Postgres cluster running while he climbs. -- reached from Bob
   (Designs the interfaces people ask questions through.), Carol (Writes the
   statistics reports finance lives on.)


kept: ['Alice', 'Bob', 'Carol', 'Dave'] ['Alice -friend-> Bob', 'Alice -friend-> Carol', 'Bob -friend-> Dave', 'Carol -friend-> Dave']


Read the two candidate sets. On its own properties, Dave's bio covers exactly as
much of the query as Carol's, and the tie breaks by id — Carol survives, and the
subgraph is one edge.

Reading how he was reached moves Dave ahead: he is the node **two routes
converge on**, and one of them runs through Carol, who writes the statistics
reports. `.paths` is what puts her sentence in his document — canonically
ordered, so the same graph always builds the same document.

And the fan-in is intact. Dave survived as a *unit*, so **both** in-edges are
reported, not just the one that scored best:

In [11]:
assert "Bob -friend-> Dave" in arrows(by_route)
assert "Carol -friend-> Dave" in arrows(by_route)
print(arrows(by_route))

['Alice -friend-> Bob', 'Alice -friend-> Carol', 'Bob -friend-> Dave', 'Carol -friend-> Dave']


That is not a courtesy. A reranked traversal runs as *probe → rerank → the
**ordinary** traversal with each step's survivors pinned*, so the edges are still
derived from the walk exactly as they always were. `Hop(near=, keep=N)` already
narrows a step to a subset; pinning narrows it to a subset the reranker chose
instead. Fan-in, multi-hop edge reconstruction and dead-end pruning are
untouched, because nothing about how they are computed changed.

**Fan-in and the bill.** By default one document is built per distinct node,
quoting every route that reached it — cost `|nodes|`, and the reranker sees all
the evidence at once. `per_path=True` builds one document per *(node, route)* and
takes the **max** over them, so one strong route is enough to keep a node:

In [12]:
per_path = graph.traverse(
    Start(where={"name": "Alice"}),
    Hop(via={"kind": "friend"}, hops=(1, 2), near=Near("interests", QUESTION), keep=1,
        rerank=Rerank(show, document_from=ROUTED, candidates=5, per_path=True)))
print("kept:", names(per_path))

 * Bob: Designs the interfaces people ask questions through. -- reached from
   Alice (Runs the platform; has not touched a query plan since 2019.)
 * Carol: Writes the statistics reports finance lives on. -- reached from
   Alice (Runs the platform; has not touched a query plan since 2019.)
 * Dave: Left a Postgres cluster running while he climbs. -- reached from Bob
   (Designs the interfaces people ask questions through.)
 * Dave: Left a Postgres cluster running while he climbs. -- reached from
   Carol (Writes the statistics reports finance lives on.)


kept: ['Alice', 'Bob', 'Carol', 'Dave']


Dave is now two documents instead of one — which is exactly why it is opt-in: it
costs `|routes|` documents, and rerankers price per document. `max_paths`
(default 10) caps how many routes one document may quote in the default mode; a
high fan-in node would otherwise blow the provider's token limit, as a hard error
if you are lucky and a silent server-side truncation if you are not.

**Each step reranks against its own query**, since `near=` is already per-step —
a three-hop traversal can rerank for a different intent at each depth. Those
calls are **serial by nature**: hop N+1's candidates are whatever hop N left
behind, so unlike `vector_search_many()`'s they cannot be issued together.

**A traversal returns a subgraph, not a ranking.** Similarity scores have never
survived into it, and rerank scores get no exception:

In [13]:
print(sorted(by_route.nodes[0]))
assert not any("rerank_score" in node for node in by_route.nodes)

['id', 'properties']


## The refusals

Every one of these is a query that cannot mean anything, refused where it was
written rather than three layers down at execution.

In [14]:
def refused(what, thunk):
    try:
        thunk()
    except (TypeError, ValueError) as exc:
        print(f"{what}\n{type(exc).__name__}: {exc}\n")


refused("a raw vector with rerank=",
        lambda: Start(near=Near("interests", [1.0, 0.0, 0.0, 0.0]), keep=1, rerank=reader))

refused("rerank= with no near=",
        lambda: Start(where={"type": "person"}, rerank=reader))

refused("rerank= with no keep=",
        lambda: Start(near=Near("interests", QUESTION), rerank=reader))

refused("candidates below keep",
        lambda: Start(near=Near("interests", QUESTION), keep=5,
                      rerank=Rerank(relevance, document_from='.properties.bio',
                                    candidates=2)))

refused("candidates equal to keep",
        lambda: Start(near=Near("interests", QUESTION), keep=5,
                      rerank=Rerank(relevance, document_from='.properties.bio',
                                    candidates=5)))

refused("per_path=True at a Start",
        lambda: graph.traverse(Start(
            near=Near("interests", QUESTION), keep=1,
            rerank=Rerank(relevance, document_from='.properties.bio',
                          candidates=5, per_path=True))))

refused(".paths read at a Start",
        lambda: graph.traverse(Start(
            near=Near("interests", QUESTION), keep=1,
            rerank=Rerank(relevance, document_from=ROUTED, candidates=5))))

a raw vector with rerank=
ValueError: Start: rerank= needs the query as TEXT, but Near('interests', ...) was given a raw vector -- a reranker scores a query against a document by reading both, and there is nothing to read in a list of floats. Write Near('interests', text="...") and the field's own embed= turns it into the vector, so the ranking and the reranking see the same query

rerank= with no near=
ValueError: Start: rerank= reorders the candidates near= ranks -- on its own it has nothing to reorder, because a reranker scores a list it is given rather than choosing one. Add near=, or drop rerank=

rerank= with no keep=
ValueError: Start: rerank= needs keep= -- a traversal returns a SUBGRAPH, not a ranking, so the order a reranker produces is discarded and truncating is the only way it can change the result. keep= IS that truncation, and with none, rerank=Rerank(candidates=5) quietly becomes the bound instead: the step keeps 5 node(s) rather than every node it reached, which is a d

per_path=True at a Start
ValueError: Start: rerank=Rerank(per_path=True) scores one document per (node, path), but a seed has no provenance -- nothing reached it, so there is exactly one document per node here and per_path=True is the default under another name. It is a Hop's mode, where a node can read differently depending on the route that found it. Move the rerank= to the Hop that reaches these nodes, or drop per_path=True from this one

.paths read at a Start
ValueError: Start: document_from='.properties.name + ": " + .properties.bio + " -- reached from " + (.paths | map(.[-1].properties.name + " (" + .[-1].properties.bio + ")") | join(", "))' reads .paths, but a seed has no provenance -- nothing reached it, so there is no path to quote. `.paths` exists at a Hop and not at a Start. Move the rerank= to the Hop that reaches these nodes, or build the seed's document from its own properties



The first is the one worth reading twice, because it is **what reranking is**
rather than a gap to close later. A reranker scores a query against a document by
reading *both*; a list of floats is not something it can read, and no amount of
implementation work makes it one. Adding a second way to supply the query — a
`query_string=` beside the vector — would let the two disagree silently, and a
reranker scoring against the wrong string is a confidently wrong ranking that
nothing reports.

The three in the middle are one rule seen from three sides: **a traversal
returns a subgraph, so the reranker's order is thrown away and truncation is the
only thing it can change.** `keep=` is that truncation, which is why a reranked
step needs one; `candidates` has to leave it more than `keep` to choose from, or
every document is billed for a result the query would have produced anyway. On a
flat search, where the order survives into the answer, neither applies —
`vector_search(rerank=…, k=3)` over `candidates=3` still reorders and still
reports a `rerank_score`.

The last two are both about provenance: a seed was reached by nothing, so there
is exactly one document per node whatever `per_path=` says, and `.paths` has
nothing to quote — returning `null` there would quietly change every document.

**A spent provider call raises; it does not degrade.** Most retrieval stacks fall
back to the pre-rerank ordering here — a caller who asked for reranking and
silently received the retrieval stage's own ordering has a different answer with
no signal, which is the hardest kind of failure for an agent to notice. Transient failures are
retried first, on `embeddings.py`'s own policy and defaults (`retries=2`,
`backoff=0.5`, full jitter, `Retry-After` winning when the provider sent one):

In [15]:
def unavailable(query, documents):
    raise ConnectionError("rerank endpoint unreachable")


try:
    graph.vector_search(Near("interests", QUESTION), k=3,
                        rerank=Rerank(unavailable, document_from='.properties.bio',
                                      candidates=5, retries=0))
except RerankError as failed:
    print(f"RerankError: {failed}\n")
    print("__cause__:", repr(failed.__cause__))

RerankError: Rerank(unavailable, document_from='.properties.bio', candidates=5).score: the reranker call failed after 1 attempt(s) (ConnectionError: rerank endpoint unreachable) -- refusing to fall back to the pre-rerank order, which would be a different answer with no signal. Catch RerankError and re-run without rerank= if that is what you want

__cause__: ConnectionError('rerank endpoint unreachable')


The provider's own exception rides along as `__cause__`, which is what makes
degrading a decision you take **in your own code, where it is visible**:

```python
try:
    hits = graph.vector_search(near, rerank=rerank, k=10)
except RerankError as failed:
    logger.warning("reranking unavailable: %s", failed.__cause__)
    hits = graph.vector_search(near, k=10)      # similarity order, on purpose
```

## When a model writes the filter

Which fields the reranker reads is a genuine retrieval decision, and one a model
is well placed to make — unlike an embedding, which it would have to invent. So
hopai is built to accept a `document_from` it did not write, and treats every
filter as untrusted by default. What that is *not* is an invitation to run
arbitrary jq: the filter's output **is** the document, and the document is posted
to a third-party API.

`hopai.jqsafe` is the gate. It is a parser over a **total subset** of jq, and
both of its guarantees are structural rather than hopeful:

- **Soundness.** jq has no dynamic dispatch to builtins by name and no `eval`, so
  the set of functions a program can call is exactly the set of literal names
  written in it — which a parser enumerates completely. `env`, `$ENV`, `input`,
  `import` are not blacklisted strings; they are *absent from the grammar and do
  not parse*.
- **Totality.** With no `def`, `while`, `until`, `repeat`, `recurse`, `range`,
  `reduce` or `foreach`, there is no unbounded iteration, so every accepted
  program terminates. That matters because libjq holds the GIL and calls
  `abort()` on memory exhaustion: neither a hang nor an out-of-memory abort can
  be caught in-process, so they have to be made *unreachable* instead.

In [16]:
from hopai import jqsafe

for filter_text in ('env.DATABASE_URL',            # the exfiltration vector
                    'def f: f; f',                 # the uninterruptible hang
                    '[range(100000000)] | tostring'):   # the uncatchable abort
    try:
        jqsafe.validate(filter_text)
    except jqsafe.UnsafeFilter as exc:
        print(f"UnsafeFilter: {exc}\n")

UnsafeFilter: document_from: `env` is not available (offset 0) -- a document is built from the candidate row only, and this filter's output is posted to a third-party reranker -- `env` would send it your process environment. Read a property instead, e.g. `.properties.title`

UnsafeFilter: document_from: `def` is not available (offset 0) -- this subset has no user-defined functions -- `def f: f; f` is a non-terminating program that no timeout can interrupt, and having no recursion at all is what makes every accepted filter terminate

UnsafeFilter: document_from: `range` is not available (offset 1) -- it generates an arbitrarily long stream -- `[range(100000000)]` makes libjq abort() the process, which no Python `except` can catch



What is left is exactly field selection — paths, indexing and iteration, `//`,
string literals, `+`, `|`, and a small allowlist of functions (`join`, `map`,
`tostring`, `ascii_downcase`, `length`, …). Every filter in this notebook is
inside it, which is not a coincidence: **every query validates**, and there is no
way to turn that off. `Rerank` has no `trusted=`, so a filter that reaches
`vector_search(rerank=…)`, a `Start` or a `Hop` is held to the subset whoever
wrote it — by then hopai cannot tell who did. `trusted=True` is a parameter of
`build_documents()` alone: the preview call above, for a human checking a filter
in their own process, where full jq restricts nothing they cannot already do.

Closing off execution still leaves *what data leaves the building*.
`.properties.email` parses perfectly and would ship straight to a vendor, so
`validate(fields=…)` publishes which paths a filter may read — a read at or
beneath an allowed path is fine, a read above one is not, since that hands back
the siblings the allowlist exists to withhold:

In [17]:
allowed = ["properties.name", "properties.bio", "properties.tags"]

print(sorted(jqsafe.paths_read('.properties.name + ": " + .properties.bio')), "\n")
jqsafe.validate('.properties.name + ": " + .properties.bio', fields=allowed)   # fine

for over_reach in ('.properties.email', '.'):
    try:
        jqsafe.validate(over_reach, fields=allowed)
    except jqsafe.UnsafeFilter as exc:
        print(f"UnsafeFilter: {exc}\n")

['properties.bio', 'properties.name'] 

UnsafeFilter: document_from: reads `.properties.email`, which is not one of the fields this filter may see: properties.name, properties.bio, properties.tags. Read one of those, or add `properties.email` to the fields this reranker is allowed to send

UnsafeFilter: document_from: `.` reads the whole row, which is more than this filter may see -- the fields it may read are: properties.name, properties.bio, properties.tags. Read them by name, e.g. `.properties.name`



`Rerank.build_documents(candidates, fields=[…])` is where an operator hands that
allowlist to a live query — the same list, applied to every candidate, so a
filter that arrived over the wire can only read what was published. Paired with
`infer_schema()`/`GraphSchema`, the readable fields can be *enumerated* for the
model rather than guessed at: safer and easier to use correctly at the same time.

A human writing `document_from=` in their own Python writes the same language a
model does — the subset is the grammar, not a second dialect — and gets full jq
in the one place it changes nothing: `build_documents(candidates, trusted=True)`,
checking a filter against real rows in a process where they already run arbitrary
code. Run that same filter as part of a query and it is validated like any other.

## What a real one looks like

Swap the stand-in for the client you already have. Nothing else on this page
changes — the contract is the same one method, and hopai imports no provider
package to reach it:

```python
import cohere
from hopai import Rerank

rerank = Rerank(cohere.ClientV2(), model="rerank-v3.5",
                document_from='.properties.title + ": " + (.properties.summary // "")',
                candidates=50)

graph.vector_search(Near("summary", "how do nodes agree?"), rerank=rerank, k=10)
```

`Rerank(voyageai.Client(), model="rerank-2", …)`,
`Rerank(CrossEncoder("cross-encoder/ms-marco-MiniLM-L-6-v2"), …)`, anything with
`.score(query, documents)`, and the plain callable used above are the accepted
shapes. `model=` is **required** where the provider takes one and **refused**
where it does not — a silently chosen reranker is a silently different ranking.
Documents are chunked to the provider's own per-call cap (1000 for Cohere and
Voyage), and answers that come back *sorted by relevance* are re-paired by their
own `.index`, never by arrival order: zipping a relevance-sorted answer against
the documents that were sent is a plausible, confidently wrong ranking with no
error anywhere.

**What it costs.** A flat `vector_search(rerank=…)` is one statement plus one
reranker call. A reranked traversal is the ordinary traversal — 3 statements, the
walk and its two hydrations — plus **2 per reranked step**, a probe and a
hydration, and one provider call per step, serial. An async client is awaited off
the event loop (`ascore()`), and `vector_search_many(rerank=…)` takes one
`Rerank` for the whole call and spends one provider call per query — in sequence
on `Graph`, **concurrently** on `AsyncGraph`, since that call exists to turn N
round trips into one.

Over MCP the reranker is the **operator's**, configured in Python
(`serve(rerank=…, rerank_fields=[…], max_candidates=…)`); a model supplies only
`document_from` and `candidates`, against the published field list — the
MCP guide's *Reranking* section is the whole operator story.

---

Back to [09 · Vector search](09_vector_search.ipynb), or on to
`hopai/rerankers.py` and `hopai/jqsafe.py`, which argue every refusal on this
page at length.